In [2]:
import data_loader, audio_preprocessing ,utils_audio_PD_project
import os
import pandas as pd
import numpy as np

c:\Users\Marcos\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


# Saving pre-processed audio data into ".npy" files

## All audio

In [7]:
# Define root directory and output folder
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Audio_HUMV_enhanced'
output_folder = r'D:\processed_audio\balanced\HC_vs_PD_5s_with_1s_overlap_HUMV_all_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

exact_names_values = ['ta.wav', 'uy.wav','lectura texto.wav', 'fluencia categorial.wav', 'habla libre.wav', 'robo galletas.wav', 'aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav' ] 
# Iterate through 'exact_names' values
for exact_names in exact_names_values:
    print(f"Processing audio files: {exact_names}")

    # Load audio data
    df = data_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)

    # Filter and relabel data
    df = df[df['Label'].isin([0, 2])]
    df['Label'] = df['Label'].replace(2, 1)
    # List of patient IDs to remove
    patients_to_remove = ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24' ,'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
    print("Removing patients:", patients_to_remove)
    # Remove duplicates just in case
    patients_to_remove = list(set(patients_to_remove))
    # Filter the dataframe
    df_filtered = df[~df['Patient'].isin(patients_to_remove)].copy()

    value_counts = df_filtered['Label'].value_counts()
    print(value_counts)

    # Preprocess and split audio
    audio_segments, labels_np, patient_ids = audio_preprocessing.execute_preprocess_and_split(
        df_filtered, start_time = 0, chunk_duration = 5, max_duration = None, target_sr = 16000, 
        remove_silence = True, top_db=25, silence_duration= 1, file_path_column='File_Path', 
        overlap= 1, min_chunk_length=0.8,
    )

    # Remove the .wav extension from exact_names
    file_name_without_extension = os.path.splitext(exact_names)[0]

    # Save arrays in the subfolder (using the modified file name)
    np.save(os.path.join(output_folder, f"audio_segments_5s_with_1s_overlap_{file_name_without_extension}.npy"), audio_segments)
    np.save(os.path.join(output_folder, f"labels_5s_with_1s_overlap_{file_name_without_extension}.npy"), labels_np)
    np.save(os.path.join(output_folder, f"patient_ids_5s_with_1s_overlap_{file_name_without_extension}.npy"), patient_ids)


    print(f"Arrays saved successfully in {output_folder}!")

print("All audio files processed and saved.")

Processing audio files: ta.wav
Loaded 116 audio files.
Label distribution:
Label
0    50
1    34
2    32
Name: count, dtype: int64
Removing patients: ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24', 'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
Label
0    44
1    29
Name: count, dtype: int64
Total chunks generated: 145
Arrays saved successfully in D:\processed_audio\balanced\HC_vs_PD_5s_with_1s_overlap_HUMV_all_audio!
Processing audio files: uy.wav
Loaded 117 audio files.
Label distribution:
Label
0    50
1    34
2    33
Name: count, dtype: int64
Removing patients: ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24', 'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
Label
0    44
1    29
Name: count, dtype: int64
Total chunks generated: 94
Arrays saved successfully in D:\processed_audio\balanced\HC_vs_PD_5s_with_1s_overlap_HUMV_all_audio!
Processing audio files: lectura texto.wav
Loaded 113 audio files.
Label

## Mean audio

Audio files will be as long as the mean of the audio

In [9]:
# Define root directory and output folder
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Audio_HUMV_enhanced'
output_folder = r'D:\processed_audio\balanced\HC_vs_PD_5s_with_1s_overlap_HUMV_mean_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    ("1 audio of 5 seconds", ['aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav', 'ta.wav', 'uy.wav', 'vocal.wav'], 1, 5, 6),
    ("6 audios of 5 seconds", ['habla libre.wav', 'robo galletas.wav'], 0, 5, 35),
    ("10 audios of 5 seconds", ['fluencia categorial.wav'], 0, 5, 60),
    ("25 audios of 5 seconds", ['lectura texto.wav'], 0, 5, 150),
]

for desc, exact_names_values, start_time, chunk_duration, max_duration in exercise_sets:
    print(f"\nProcessing set: {desc}")
    for exact_names in exact_names_values:
        print(f"  Processing audio files: {exact_names}")
        df = data_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)
        df = df[df['Label'].isin([0, 2])]
        df['Label'] = df['Label'].replace(2, 1)
        
        # List of patient IDs to remove
        patients_to_remove = ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24' ,'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
        print("Removing patients:", patients_to_remove)
        # Remove duplicates just in case
        patients_to_remove = list(set(patients_to_remove))
        # Filter the dataframe
        df_filtered = df[~df['Patient'].isin(patients_to_remove)].copy()

        value_counts = df_filtered['Label'].value_counts()
        print(value_counts) 

        audio_segments, labels_np, patient_ids = audio_preprocessing.execute_preprocess_and_split(
            df_filtered,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=max_duration,
            target_sr=16000,
            remove_silence=True,
            top_db=25,
            silence_duration=1,
            file_path_column='File_Path',
            overlap=1,
            min_chunk_length=0.8,
        )
        file_name_without_extension = os.path.splitext(exact_names)[0]
        np.save(os.path.join(output_folder, f"audio_segments_5s_with_1s_overlap_{file_name_without_extension}.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_5s_with_1s_overlap_{file_name_without_extension}.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_5s_with_1s_overlap_{file_name_without_extension}.npy"), patient_ids)
        print(f"    Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")


Processing set: 1 audio of 5 seconds
  Processing audio files: aueoi.wav
Loaded 114 audio files.
Label distribution:
Label
0    49
1    34
2    31
Name: count, dtype: int64
Removing patients: ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24', 'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
Label
0    43
1    29
Name: count, dtype: int64
Total chunks generated: 71
    Arrays saved successfully in D:\processed_audio\balanced\HC_vs_PD_5s_with_1s_overlap_HUMV_mean_audio!
  Processing audio files: ka.wav
Loaded 118 audio files.
Label distribution:
Label
0    50
1    34
2    34
Name: count, dtype: int64
Removing patients: ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24', 'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
Label
0    44
1    30
Name: count, dtype: int64
Total chunks generated: 74
    Arrays saved successfully in D:\processed_audio\balanced\HC_vs_PD_5s_with_1s_overlap_HUMV_mean_audio!
  Processing audi

# AC Audios

## All audio

In [ ]:
# Define root directory and output folder
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Audio_HUMV_enhanced'
output_folder = r'D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

exact_names_values = ['ta.wav', 'uy.wav','lectura texto.wav', 'fluencia categorial.wav', 'habla libre.wav', 'robo galletas.wav', 'aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav' ] 
# Iterate through 'exact_names' values
for exact_names in exact_names_values:
    print(f"Processing audio files: {exact_names}")

    # Load audio data
    df = data_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)

    # Filter and relabel data
    df_AC = df[df['Label'].isin([1])]
    # Keep only selected patients with DaTSCAN
    keep_patients = [
        "HUMV_AC_1", "HUMV_AC_3", "HUMV_AC_4", "HUMV_AC_6", "HUMV_AC_9", "HUMV_AC_10",
        "HUMV_AC_11", "HUMV_AC_12", "HUMV_AC_17", "HUMV_AC_18", "HUMV_AC_19",
        "HUMV_AC_20", "HUMV_AC_21", "HUMV_AC_22", "HUMV_AC_23",
        "HUMV_AC_25", "HUMV_AC_30", "HUMV_AC_33", "HUMV_AC_34"
    ] # "HUMV_AC_24", "HUMV_AC_32" estos no los mantenemos al ser muy jóvenes
    df_AC = df_AC[df_AC['Patient'].isin(keep_patients)]
    # Manually set label 0 for specific patients
    patients_to_change = ["HUMV_AC_1", "HUMV_AC_3", "HUMV_AC_25", "HUMV_AC_33"]
    df_AC.loc[df_AC['Patient'].isin(patients_to_change), 'Label'] = 0

    # Preprocess and split audio
    audio_segments, labels_np, patient_ids = audio_preprocessing.execute_preprocess_and_split(
        df_AC, start_time = 0, chunk_duration = 5, max_duration = None, target_sr = 16000, 
        remove_silence = True, top_db=25, silence_duration= 1, file_path_column='File_Path', 
        overlap= 1, min_chunk_length=0.8
    )

    # Remove the .wav extension from exact_names
    file_name_without_extension = os.path.splitext(exact_names)[0]

    # Save arrays in the subfolder (using the modified file name)
    np.save(os.path.join(output_folder, f"AC_audio_segments_5s_with_1s_overlap_{file_name_without_extension}.npy"), audio_segments)
    np.save(os.path.join(output_folder, f"AC_labels_5s_with_1s_overlap_{file_name_without_extension}.npy"), labels_np)
    np.save(os.path.join(output_folder, f"AC_patient_ids_5s_with_1s_overlap_{file_name_without_extension}.npy"), patient_ids)


    print(f"Arrays saved successfully in {output_folder}!")

print("All audio files processed and saved.")

Processing audio files: ta.wav
Loaded 116 audio files.
Label distribution:
Label
0    50
1    34
2    32
Name: count, dtype: int64
Total chunks generated: 37
Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_audio!
Processing audio files: uy.wav
Loaded 117 audio files.
Label distribution:
Label
0    50
1    34
2    33
Name: count, dtype: int64
Total chunks generated: 25
Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_audio!
Processing audio files: lectura texto.wav
Loaded 113 audio files.
Label distribution:
Label
0    50
1    34
2    29
Name: count, dtype: int64
Total chunks generated: 714
Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_audio!
Processing audio files: fluencia categorial.wav
Loaded 116 audio files.
Label distribution:
Label
0    50
1    34
2    32
Name: count, dtype: int64
Total chunks generated: 147
Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_a

## Mean audio

In [3]:
# Define root directory and output folder
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Audio_HUMV_enhanced'
output_folder = r'D:\processed_audio\AC_5s_with_1s_overlap_HUMV_mean_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    ("1 audio of 5 seconds", ['aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav', 'ta.wav', 'uy.wav', 'vocal.wav'], 1, 5, 6),
    ("6 audios of 5 seconds", ['habla libre.wav', 'robo galletas.wav'], 0, 5, 35),
    ("10 audios of 5 seconds", ['fluencia categorial.wav'], 0, 5, 60),
    ("25 audios of 5 seconds", ['lectura texto.wav'], 0, 5, 150),
]

for desc, exact_names_values, start_time, chunk_duration, max_duration in exercise_sets:
    print(f"\nProcessing set: {desc}")
    for exact_names in exact_names_values:
        print(f"  Processing audio files: {exact_names}")
        # Load audio data
        df = data_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)
        # Filter and relabel data
        df_AC = df[df['Label'].isin([1])]

        # Keep only selected patients with DaTSCAN
        keep_patients = [
            "HUMV_AC_1", "HUMV_AC_3", "HUMV_AC_4", "HUMV_AC_6", "HUMV_AC_9", "HUMV_AC_10",
            "HUMV_AC_11", "HUMV_AC_12", "HUMV_AC_17", "HUMV_AC_18", "HUMV_AC_19",
            "HUMV_AC_20", "HUMV_AC_21", "HUMV_AC_22", "HUMV_AC_23",
            "HUMV_AC_25", "HUMV_AC_30", "HUMV_AC_33", "HUMV_AC_34"
        ] # "HUMV_AC_24", "HUMV_AC_32" estos no los mantenemos al ser muy jóvenes
        df_AC = df_AC[df_AC['Patient'].isin(keep_patients)]
        
        # Manually set label 0 for specific patients
        patients_to_change = ["HUMV_AC_1", "HUMV_AC_3", "HUMV_AC_25", "HUMV_AC_33"]
        df_AC.loc[df_AC['Patient'].isin(patients_to_change), 'Label'] = 0

        audio_segments, labels_np, patient_ids = audio_preprocessing.execute_preprocess_and_split(
            df_AC,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=max_duration,
            target_sr=16000,
            remove_silence=True,
            top_db=25,
            silence_duration=1,
            file_path_column='File_Path',
            overlap=1,
            min_chunk_length=0.8,
        )
        file_name_without_extension = os.path.splitext(exact_names)[0]
        np.save(os.path.join(output_folder, f"audio_segments_5s_with_1s_overlap_{file_name_without_extension}.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_5s_with_1s_overlap_{file_name_without_extension}.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_5s_with_1s_overlap_{file_name_without_extension}.npy"), patient_ids)
        print(f"    Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")


Processing set: 1 audio of 5 seconds
  Processing audio files: aueoi.wav
Loaded 114 audio files.
Label distribution:
Label
0    49
1    34
2    31
Name: count, dtype: int64
Total chunks generated: 19
    Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_mean_audio!
  Processing audio files: ka.wav
Loaded 118 audio files.
Label distribution:
Label
0    50
1    34
2    34
Name: count, dtype: int64
Total chunks generated: 19
    Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_mean_audio!
  Processing audio files: pa.wav
Loaded 116 audio files.
Label distribution:
Label
0    50
1    34
2    32
Name: count, dtype: int64
Total chunks generated: 19
    Arrays saved successfully in D:\processed_audio\AC_5s_with_1s_overlap_HUMV_mean_audio!
  Processing audio files: patachaka.wav
Loaded 117 audio files.
Label distribution:
Label
0    50
1    34
2    33
Name: count, dtype: int64
Total chunks generated: 19
    Arrays saved successfully in D:\pro

# Checking labels and ids

In [4]:
# Define directories for both datasets
dir_1 = r'D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_audio'

# Load both datasets
all_patient_ids_HUMV, all_labels_HUMV, all_audio_segments_HUMV, _ = utils_audio_PD_project.load_processed_audio_dataset(dir_1, 0, GRL=False, exercise_names = ["uy", "ta"]) #, "pa", "ka", "pataka", "patachaka", "aueoi", "lectura texto", "habla libre", "robo galletas", "fluencia categorial"

# all_patient_ids, all_labels, all_audio_segments, _ = utils_audio_PD_project.load_processed_audio_dataset(dir_1, 0, GRL=False, exercise_names = ["uy"])


# Check final dataset sizes
print(f"Total audio segments: {len(all_audio_segments_HUMV)}")
print(f"Total labels: {len(all_labels_HUMV)}")
print(f"Total patient IDs: {len(all_patient_ids_HUMV)}")

# Validate label distribution
print(f"Final label distribution: {np.bincount(all_labels_HUMV)}")

Loading dataset from: D:\processed_audio\AC_5s_with_1s_overlap_HUMV_all_audio
Selected exercises: ['uy', 'ta']
Sorted Patient ID files: ['AC_patient_ids_5s_with_1s_overlap_ta.npy', 'AC_patient_ids_5s_with_1s_overlap_uy.npy']
Sorted Label files: ['AC_labels_5s_with_1s_overlap_ta.npy', 'AC_labels_5s_with_1s_overlap_uy.npy']
Sorted Audio files: ['AC_audio_segments_5s_with_1s_overlap_ta.npy', 'AC_audio_segments_5s_with_1s_overlap_uy.npy']
Total audio segments: 62
Total labels: 62
Total patient IDs: 62
Final label distribution: [13 49]


In [5]:
all_patient_ids_HUMV

array(['HUMV_AC_1', 'HUMV_AC_1', 'HUMV_AC_10', 'HUMV_AC_10', 'HUMV_AC_11',
       'HUMV_AC_11', 'HUMV_AC_12', 'HUMV_AC_12', 'HUMV_AC_17',
       'HUMV_AC_17', 'HUMV_AC_18', 'HUMV_AC_18', 'HUMV_AC_19',
       'HUMV_AC_19', 'HUMV_AC_20', 'HUMV_AC_20', 'HUMV_AC_21',
       'HUMV_AC_21', 'HUMV_AC_22', 'HUMV_AC_22', 'HUMV_AC_23',
       'HUMV_AC_23', 'HUMV_AC_25', 'HUMV_AC_25', 'HUMV_AC_3', 'HUMV_AC_3',
       'HUMV_AC_30', 'HUMV_AC_30', 'HUMV_AC_33', 'HUMV_AC_34',
       'HUMV_AC_34', 'HUMV_AC_4', 'HUMV_AC_4', 'HUMV_AC_6', 'HUMV_AC_6',
       'HUMV_AC_9', 'HUMV_AC_9', 'HUMV_AC_1', 'HUMV_AC_1', 'HUMV_AC_10',
       'HUMV_AC_10', 'HUMV_AC_11', 'HUMV_AC_12', 'HUMV_AC_12',
       'HUMV_AC_17', 'HUMV_AC_18', 'HUMV_AC_19', 'HUMV_AC_20',
       'HUMV_AC_21', 'HUMV_AC_22', 'HUMV_AC_23', 'HUMV_AC_25',
       'HUMV_AC_25', 'HUMV_AC_3', 'HUMV_AC_30', 'HUMV_AC_33',
       'HUMV_AC_34', 'HUMV_AC_4', 'HUMV_AC_4', 'HUMV_AC_6', 'HUMV_AC_6',
       'HUMV_AC_9'], dtype='<U10')

In [20]:
all_patient_ids_HUMV

array(['HUMV_HC_1', 'HUMV_HC_1', 'HUMV_HC_10', 'HUMV_HC_10', 'HUMV_HC_11',
       'HUMV_HC_11', 'HUMV_HC_12', 'HUMV_HC_12', 'HUMV_HC_13',
       'HUMV_HC_13', 'HUMV_HC_14', 'HUMV_HC_14', 'HUMV_HC_15',
       'HUMV_HC_15', 'HUMV_HC_16', 'HUMV_HC_16', 'HUMV_HC_17',
       'HUMV_HC_17', 'HUMV_HC_18', 'HUMV_HC_18', 'HUMV_HC_19',
       'HUMV_HC_19', 'HUMV_HC_2', 'HUMV_HC_2', 'HUMV_HC_20', 'HUMV_HC_20',
       'HUMV_HC_21', 'HUMV_HC_21', 'HUMV_HC_22', 'HUMV_HC_22',
       'HUMV_HC_23', 'HUMV_HC_23', 'HUMV_HC_24', 'HUMV_HC_24',
       'HUMV_HC_25', 'HUMV_HC_25', 'HUMV_HC_26', 'HUMV_HC_26',
       'HUMV_HC_3', 'HUMV_HC_3', 'HUMV_HC_4', 'HUMV_HC_4', 'HUMV_HC_5',
       'HUMV_HC_5', 'HUMV_HC_6', 'HUMV_HC_6', 'HUMV_HC_7', 'HUMV_HC_7',
       'HUMV_HC_8', 'HUMV_HC_8', 'HUMV_HC_9', 'HUMV_HC_9', 'HUMV_NFC_1',
       'HUMV_NFC_1', 'HUMV_NFC_10', 'HUMV_NFC_10', 'HUMV_NFC_11',
       'HUMV_NFC_11', 'HUMV_NFC_12', 'HUMV_NFC_12', 'HUMV_NFC_13',
       'HUMV_NFC_13', 'HUMV_NFC_14', 'HUMV_NFC_14', 'HUMV_N